# ZuCo sentiment from classical EEG features

Classical stats of the raw EEG plus ZuCo's precomputed band-power means, then a
linear classifier. A plain CPU runtime is fine here, no GPU needed.
Mount Drive (the `.mat` files are large and live there), point `MAT_DIR` at them
and `RESULTS_DIR` at where you want the output, then run top to bottom.

In [ ]:
# clone (or pull) the code, install deps
import os
REPO = 'zuco-eeg-classical-features'
if not os.path.exists(REPO):
    !git clone https://github.com/parmisbathaeiyan/zuco-eeg-classical-features.git $REPO
else:
    !cd $REPO && git pull --ff-only
%cd $REPO
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder with the EEG results*_SR.mat files
MAT_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'

# where to save results (on Drive so they survive a runtime reset). Created if
# it doesn't exist; change to e.g. 'results' to keep them on the Colab disk.
RESULTS_DIR = '/content/drive/MyDrive/Thesis/Results/zuco_eeg_classical_features'

In [ ]:
# 1. extract features -> features/<SUBJ>.npz  (slow; resumable, skips cached)
!python extract_features.py --mat-dir "$MAT_DIR" --out-dir features

In [ ]:
# 2. classify: subject-specific + pooled, results written under RESULTS_DIR
!python run.py --features-dir features --output-dir "$RESULTS_DIR" --classifier logreg

In [ ]:
# 3. tables + report -> RESULTS_DIR/tables/{summary.csv, per_subject_*.csv, report.md}
#    redirect the CLI's text output to /dev/null and show only the rendered report
!python make_tables.py --results-dir "$RESULTS_DIR" > /dev/null

from IPython.display import Markdown, display
display(Markdown(open(os.path.join(RESULTS_DIR, 'tables', 'report.md')).read()))

In [ ]:
# the plots run.py saved
from IPython.display import Image, display
for name in ['plots/subject_accuracy_logreg.png',
             'plots/cm_pooled_logreg.png']:
    p = os.path.join(RESULTS_DIR, name)
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# 4. feature-label association: pooled + per-subject (ANOVA F / Spearman)
!python analyze_features.py --features-dir features --output-dir "$RESULTS_DIR" > /dev/null

from IPython.display import Markdown, Image, display
A = os.path.join(RESULTS_DIR, 'associations')
display(Markdown(open(os.path.join(A, 'associations.md')).read()))      # pooled
display(Markdown(open(os.path.join(A, 'subject_report.md')).read()))    # per-subject
for name in ['by_stat.png', 'by_band.png', 'top_features.png', 'subject_by_stat.png']:
    p = os.path.join(A, name)
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# 5. spatial montage: per-channel association mapped onto the scalp, one panel per family
#    (needs feature_association.csv from the cell above, and zuco_montage.npz in the repo)
!python plot_montage.py --results-dir "$RESULTS_DIR" --montage zuco_montage.npz > /dev/null

from IPython.display import Image, display
for name in ['montage/montage_stats_f_score.png', 'montage/montage_bands_f_score.png']:
    p = os.path.join(RESULTS_DIR, name)
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# (optional) channel-averaged comparison: 24 family means instead of 2520 per-channel.
# Tests whether averaging over channels lifts a diffuse signal out of the noise.
CHAVG_DIR = RESULTS_DIR + '_chavg'
!python run.py --features-dir features --output-dir "$CHAVG_DIR" --classifier logreg --channel-avg > /dev/null
!python make_tables.py --results-dir "$CHAVG_DIR" > /dev/null
!python analyze_features.py --features-dir features --output-dir "$CHAVG_DIR" --channel-avg > /dev/null

from IPython.display import Markdown, display
display(Markdown("## Channel-averaged (24 features)"))
display(Markdown(open(os.path.join(CHAVG_DIR, 'tables', 'report.md')).read()))
display(Markdown(open(os.path.join(CHAVG_DIR, 'associations', 'associations.md')).read()))